In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql import functions as F

tabla_origen = (
    "catalog_au.silver.sfc_indicadores_fact_transform"
)

tabla_destino = (
    "catalog_au.golden.fact_indicadores_financieros"
)

df_fact_load = spark.table(tabla_origen)

print("Registros para cargar:", df_fact_load.count())

df_fact_load.printSchema()

In [0]:
%sql
CREATE TABLE IF NOT EXISTS catalog_au.golden.fact_indicadores_financieros (

    sk_fact BIGINT GENERATED ALWAYS AS IDENTITY,

    id_indicador STRING NOT NULL,

    sk_entidad BIGINT NOT NULL,

    sk_concepto BIGINT NOT NULL,

    sk_tiempo INT NOT NULL,

    valor_indicador DECIMAL(38,18),

    unidad_medida STRING,

    fecha_corte DATE,

    archivo_origen STRING,

    fecha_ingesta TIMESTAMP,

    fecha_procesamiento TIMESTAMP,

    fecha_creacion TIMESTAMP,

    fecha_actualizacion TIMESTAMP

)
USING DELTA;

In [0]:
df_fact_golden = spark.table(
    "catalog_au.golden.fact_indicadores_financieros"
)

print("Esquema Golden:")
df_fact_golden.printSchema()

print(
    "Registros actuales:",
    df_fact_golden.count()
)

In [0]:
from pyspark.sql import functions as F

df_validacion_decimal = df_fact_load.agg(
    F.count("*").alias("total_registros"),

    F.sum(
        F.when(
            F.col("valor_indicador").cast("decimal(28,6)").isNull()
            & F.col("valor_indicador").isNotNull(),
            1
        ).otherwise(0)
    ).alias("valores_fuera_rango"),

    F.sum(
        F.when(
            F.col("valor_indicador") !=
            F.col("valor_indicador").cast("decimal(28,6)"),
            1
        ).otherwise(0)
    ).alias("valores_con_redondeo"),

    F.max(
        F.abs(F.col("valor_indicador"))
    ).alias("valor_maximo_absoluto")
)

display(df_validacion_decimal)

In [0]:
display(
    df_fact_load
    .groupBy("fecha_corte", "sk_tiempo")
    .agg(
        F.count("*").alias("total_indicadores"),
        F.countDistinct("sk_entidad").alias("entidades"),
        F.countDistinct("sk_concepto").alias("conceptos")
    )
    .orderBy("fecha_corte")
)

In [0]:
%sql
CREATE OR REPLACE TABLE catalog_au.golden.fact_indicadores_financieros (

    id_hecho STRING NOT NULL,

    sk_entidad BIGINT NOT NULL,

    sk_concepto BIGINT NOT NULL,

    sk_tiempo INT NOT NULL,

    fecha_corte DATE NOT NULL,

    valor_indicador DECIMAL(38,18),

    unidad_medida STRING,

    archivo_origen STRING,

    fecha_ingesta TIMESTAMP,

    fecha_procesamiento TIMESTAMP

)
USING DELTA;

In [0]:
df_fact_golden = spark.table(
    "catalog_au.golden.fact_indicadores_financieros"
)

df_fact_golden.printSchema()

print(
    "Registros Golden:",
    df_fact_golden.count()
)

In [0]:
from pyspark.sql import functions as F

tabla_origen = "catalog_au.silver.sfc_indicadores_fact_transform"
tabla_destino = "catalog_au.golden.fact_indicadores_financieros"

df_fact_load = spark.table(tabla_origen)

control = df_fact_load.agg(
    F.count("*").alias("total"),
    F.countDistinct("id_indicador").alias("ids_unicos"),
    F.sum(
        F.when(
            F.col("id_indicador").isNull() |
            F.col("sk_entidad").isNull() |
            F.col("sk_concepto").isNull() |
            F.col("sk_tiempo").isNull() |
            F.col("fecha_corte").isNull(),
            1
        ).otherwise(0)
    ).alias("claves_nulas")
).first()

assert control["total"] == 130130, "Cantidad inesperada de registros"
assert control["ids_unicos"] == control["total"], "Existen IDs duplicados"
assert control["claves_nulas"] == 0, "Existen claves obligatorias nulas"

print("VALIDACIÓN PRE-LOAD: OK")
print("Registros para cargar:", control["total"])
print("Registros actuales en Golden:", spark.table(tabla_destino).count())

df_fact_load.createOrReplaceTempView("vw_fact_indicadores_load")

In [0]:
spark.sql("""
MERGE INTO catalog_au.golden.fact_indicadores_financieros AS target

USING vw_fact_indicadores_load AS source

ON target.id_hecho = source.id_indicador

WHEN MATCHED AND NOT (
    target.sk_entidad <=> source.sk_entidad
    AND target.sk_concepto <=> source.sk_concepto
    AND target.sk_tiempo <=> source.sk_tiempo
    AND target.fecha_corte <=> source.fecha_corte
    AND target.valor_indicador <=> source.valor_indicador
    AND target.unidad_medida <=> source.unidad_medida
    AND target.archivo_origen <=> source.archivo_origen
    AND target.fecha_ingesta <=> source.fecha_ingesta
)
THEN UPDATE SET
    target.sk_entidad = source.sk_entidad,
    target.sk_concepto = source.sk_concepto,
    target.sk_tiempo = source.sk_tiempo,
    target.fecha_corte = source.fecha_corte,
    target.valor_indicador = source.valor_indicador,
    target.unidad_medida = source.unidad_medida,
    target.archivo_origen = source.archivo_origen,
    target.fecha_ingesta = source.fecha_ingesta,
    target.fecha_procesamiento = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
    id_hecho,
    sk_entidad,
    sk_concepto,
    sk_tiempo,
    fecha_corte,
    valor_indicador,
    unidad_medida,
    archivo_origen,
    fecha_ingesta,
    fecha_procesamiento
)
VALUES (
    source.id_indicador,
    source.sk_entidad,
    source.sk_concepto,
    source.sk_tiempo,
    source.fecha_corte,
    source.valor_indicador,
    source.unidad_medida,
    source.archivo_origen,
    source.fecha_ingesta,
    current_timestamp()
)
""")

print("MERGE INTO Golden ejecutado correctamente")

In [0]:
df_resultado = spark.sql("""
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_hecho) AS ids_unicos,
    COUNT(DISTINCT sk_entidad) AS entidades,
    COUNT(DISTINCT sk_concepto) AS conceptos,
    COUNT(DISTINCT sk_tiempo) AS periodos,
    SUM(
        CASE
            WHEN sk_entidad IS NULL
              OR sk_concepto IS NULL
              OR sk_tiempo IS NULL
            THEN 1 ELSE 0
        END
    ) AS claves_nulas
FROM catalog_au.golden.fact_indicadores_financieros
""")

display(df_resultado)

In [0]:
df_antes = spark.sql("""
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_hecho) AS ids_unicos,
    MAX(fecha_procesamiento) AS ultima_actualizacion
FROM catalog_au.golden.fact_indicadores_financieros
""")

display(df_antes)

In [0]:
df_despues = spark.sql("""
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT id_hecho) AS ids_unicos,
    MAX(fecha_procesamiento) AS ultima_actualizacion
FROM catalog_au.golden.fact_indicadores_financieros
""")

display(df_despues)

In [0]:
%sql
SELECT
    c.codigo_concepto,
    c.nombre_concepto,
    c.nivel_jerarquia,
    c.categoria_financiera,
    COUNT(*) AS total_registros,
    COUNT(DISTINCT f.sk_entidad) AS versiones_entidad,
    COUNT(DISTINCT f.sk_tiempo) AS periodos

FROM catalog_au.golden.fact_indicadores_financieros f

INNER JOIN catalog_au.golden.dim_concepto_financiero c
    ON f.sk_concepto = c.sk_concepto

WHERE
    LOWER(c.nombre_concepto) LIKE '%activo%'
    OR LOWER(c.nombre_concepto) LIKE '%pasivo%'
    OR LOWER(c.nombre_concepto) LIKE '%patrimonio%'
    OR LOWER(c.nombre_concepto) LIKE '%utilidad%'

GROUP BY
    c.codigo_concepto,
    c.nombre_concepto,
    c.nivel_jerarquia,
    c.categoria_financiera

ORDER BY
    c.nombre_concepto;

In [0]:
%sql
WITH balance AS (
    SELECT
        f.sk_entidad,
        f.fecha_corte,

        MAX(CASE
            WHEN c.codigo_concepto = 'SFC_FILA_0014'
            THEN f.valor_indicador
        END) AS activos,

        MAX(CASE
            WHEN c.codigo_concepto = 'SFC_FILA_0101'
            THEN f.valor_indicador
        END) AS pasivos,

        MAX(CASE
            WHEN c.codigo_concepto = 'SFC_FILA_0168'
            THEN f.valor_indicador
        END) AS patrimonio

    FROM catalog_au.golden.fact_indicadores_financieros f

    INNER JOIN catalog_au.golden.dim_concepto_financiero c
        ON f.sk_concepto = c.sk_concepto

    WHERE c.codigo_concepto IN (
        'SFC_FILA_0014',
        'SFC_FILA_0101',
        'SFC_FILA_0168'
    )

    GROUP BY
        f.sk_entidad,
        f.fecha_corte
)

SELECT
    fecha_corte,
    COUNT(*) AS total_entidades,
    SUM(
        CASE
            WHEN activos IS NULL
              OR pasivos IS NULL
              OR patrimonio IS NULL
            THEN 1 ELSE 0
        END
    ) AS balances_incompletos,
    SUM(
        CASE
            WHEN activos IS NOT NULL
             AND pasivos IS NOT NULL
             AND patrimonio IS NOT NULL
             AND ABS(activos - pasivos - patrimonio) > 0.01
            THEN 1 ELSE 0
        END
    ) AS balances_con_diferencia,
    MAX(
        ABS(activos - pasivos - patrimonio)
    ) AS diferencia_maxima

FROM balance
GROUP BY fecha_corte
ORDER BY fecha_corte;

In [0]:
%sql
CREATE OR REPLACE VIEW
catalog_au.golden.vw_powerbi_indicadores_financieros
AS

SELECT
    f.id_hecho,

    e.codigo_entidad,
    e.nombre_entidad,
    e.tipo_entidad,

    c.codigo_concepto,
    c.nombre_concepto,
    c.categoria_financiera,

    t.anio,
    t.mes,
    t.nombre_mes,
    f.fecha_corte,

    f.valor_indicador,
    f.unidad_medida

FROM catalog_au.golden.fact_indicadores_financieros f

INNER JOIN catalog_au.golden.dim_entidad e
    ON f.sk_entidad = e.sk_entidad

INNER JOIN catalog_au.golden.dim_concepto_financiero c
    ON f.sk_concepto = c.sk_concepto

INNER JOIN catalog_au.golden.dim_tiempo t
    ON f.sk_tiempo = t.sk_tiempo;

In [0]:
%sql
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT codigo_entidad) AS entidades,
    COUNT(DISTINCT codigo_concepto) AS conceptos,
    COUNT(DISTINCT fecha_corte) AS periodos
FROM catalog_au.golden.vw_powerbi_indicadores_financieros;